In [1]:
from pathlib import Path
import numpy as np
import wfdb # type: ignore
import matplotlib.pyplot as plt
import pandas as pd
import csv
import pywt # type: ignore
from scipy.signal import hilbert
from scipy.signal import find_peaks
import ast

In [2]:
def strip_wfdb_suffix(p: Path) -> Path:

    while p.suffix in (".dat", ".hea"):
        p = p.with_suffix("")
    return p

In [3]:
def detectar_picos_R(derivacion_I, wavelet, nivel):
    coeffs = pywt.wavedec(derivacion_I, wavelet, level=nivel)
    coeffs_qrs = [np.zeros_like(c) for c in coeffs]
    coeffs_qrs[4] = coeffs[4]  # d5
    coeffs_qrs[5] = coeffs[5]  # d4
    qrs_signal = pywt.waverec(coeffs_qrs, 'db6')
    qrs_signal = qrs_signal[:len(derivacion_I)]
    analytic_signal = hilbert(qrs_signal)
    envelope = np.abs(analytic_signal)
    umbral = np.mean(envelope) + 1.0 * np.std(envelope)
    peaks, _ = find_peaks(envelope, height=umbral, distance=200)
    return peaks

In [4]:
def extraer_latidos(paciente,derivacion,pre,post):
    latidos_recortados=[]
    for latidos in paciente:
        if latidos-pre>=0 and latidos+post<len(derivacion):
            latido=derivacion[latidos-pre:latidos+post]
            latidos_recortados.append(latido)
    latidos_recortados=np.array(latidos_recortados)
    return latidos_recortados

def pintar_latidos(latidos, paciente):
    tiempo = np.linspace(-0.2, 0.4, pre+post)  # eje de tiempo
    
    plt.figure(figsize=(10,6))  # figura única por paciente

    for beat in latidos:
        plt.plot(tiempo, beat, alpha=0.6)  # todos los latidos en la misma gráfica

    plt.axvline(0, color='red', linestyle='--', label='R peak')
    plt.title("Latidos alineados en R del paciente {}".format(paciente))
    plt.xlabel("Tiempo (s)")
    plt.ylabel("Amplitud")
    plt.legend()
    plt.show()

In [5]:
def construir_matriz_paciente(latidos,max_latidos,longitud_latido):
    matriz_paciente = np.zeros((max_latidos, longitud_latido))
    for i in range(min(len(latidos), max_latidos)):
        latido = latidos[i]
        
        # por seguridad si alguna ventana sale rara
        if len(latido) >= longitud_latido:
            matriz_paciente[i] = latido[:longitud_latido]
        else:
            matriz_paciente[i, :len(latido)] = latido
    return matriz_paciente

def construir_vector_etiqueta(etiqueta, enfermedades):
    etiqueta = ast.literal_eval(etiqueta)
    y_vector = np.zeros(len(enfermedades))
    
    for i, clase in enumerate(enfermedades):
        if clase in etiqueta:
            y_vector[i] = etiqueta[clase] / 100.0
    return y_vector

def vector_etiqueta_grupo(etiqueta, enfermedad_grupo,ENFERMEDADES):
    etiqueta = ast.literal_eval(etiqueta)
    y_vector = np.zeros(len(ENFERMEDADES))
    for enfermedad,prob in etiqueta.items():
        if prob>0:
            grupo=enfermedad_grupo.get(enfermedad, None)
            if grupo in ENFERMEDADES:
                idx = ENFERMEDADES.index(grupo)
                y_vector[idx] = 1
    return y_vector
    

In [6]:
df_pacientes=pd.read_csv("C:\\Users\\Ana\\Documents\\4º GITT+BA (2025-2026)\\TFG\\TFG\\ptbxl_database.csv")
df_scp=pd.read_csv("C:\\Users\\Ana\\Documents\\4º GITT+BA (2025-2026)\\TFG\\TFG\\scp_statements.csv")

In [7]:
df_pacientes

,ecg_id,patient_id,age,sex,height,weight,nurse,site,device,recording_date,...,validated_by_human,baseline_drift,static_noise,burst_noise,electrodes_problems,extra_beats,pacemaker,strat_fold,filename_lr,filename_hr
0,1,15709.0,56.0,1,NaN,63.0,2.0,0.0,CS-12 E,1984-11-09 09:17:34,...,True,NaN,", I-V1,",NaN,NaN,NaN,NaN,3,records100/00000/00001_lr,records500/00000/00001_hr
1,2,13243.0,19.0,0,NaN,70.0,2.0,0.0,CS-12 E,1984-11-14 12:55:37,...,True,NaN,NaN,NaN,NaN,NaN,NaN,2,records100/00000/00002_lr,records500/00000/00002_hr
2,3,20372.0,37.0,1,NaN,69.0,2.0,0.0,CS-12 E,1984-11-15 12:49:10,...,True,NaN,NaN,NaN,NaN,NaN,NaN,5,records100/00000/00003_lr,records500/00000/00003_hr
3,4,17014.0,24.0,0,NaN,82.0,2.0,0.0,CS-12 E,1984-11-15 13:44:57,...,True,", II,III,AVF",NaN,NaN,NaN,NaN,NaN,3,records100/00000/00004_lr,records500/00000/00004_hr
4,5,17448.0,19.0,1,NaN,70.0,2.0,0.0,CS-12 E,1984-11-17 10:43:15,...,True,", III,AVR,AVF",NaN,NaN,NaN,NaN,NaN,4,records100/00000/00005_lr,records500/00000/00005_hr
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
21794,21833,17180.0,67.0,1,NaN,NaN,1.0,2.0,AT-60 3,2001-05-31 09:14:35,...,True,NaN,", alles,",NaN,NaN,1ES,NaN,7,records100/21000/21833_lr,records500/21000/21833_hr
21795,21834,20703.0,300.0,0,NaN,NaN,1.0,2.0,AT-60 3,2001-06-05 11:33:39,...,True,NaN,NaN,NaN,NaN,NaN,NaN,4,records100/21000/21834_lr,records500/21000/21834_hr
21796,21835,19311.0,59.0,1,NaN,NaN,1.0,2.0,AT-60 3,2001-06-08 10:30:27,...,True,NaN,", I-AVR,",NaN,NaN,NaN,NaN,2,records100/21000/21835_lr,records500/21000/21835_hr
21797,21836,8873.0,64.0,1,NaN,NaN,1.0,2.0,AT-60 3,2001-06-09 18:21:49,...,True,NaN,NaN,NaN,NaN,SVES,NaN,8,records100/21000/21836_lr,records500/21000/21836_hr


In [8]:
df_pacientes = df_pacientes[(df_pacientes["age"] != df_pacientes["age"].max()) & (df_pacientes["age"] >= 18)]

In [9]:
df_pacientes

,ecg_id,patient_id,age,sex,height,weight,nurse,site,device,recording_date,...,validated_by_human,baseline_drift,static_noise,burst_noise,electrodes_problems,extra_beats,pacemaker,strat_fold,filename_lr,filename_hr
0,1,15709.0,56.0,1,NaN,63.0,2.0,0.0,CS-12 E,1984-11-09 09:17:34,...,True,NaN,", I-V1,",NaN,NaN,NaN,NaN,3,records100/00000/00001_lr,records500/00000/00001_hr
1,2,13243.0,19.0,0,NaN,70.0,2.0,0.0,CS-12 E,1984-11-14 12:55:37,...,True,NaN,NaN,NaN,NaN,NaN,NaN,2,records100/00000/00002_lr,records500/00000/00002_hr
2,3,20372.0,37.0,1,NaN,69.0,2.0,0.0,CS-12 E,1984-11-15 12:49:10,...,True,NaN,NaN,NaN,NaN,NaN,NaN,5,records100/00000/00003_lr,records500/00000/00003_hr
3,4,17014.0,24.0,0,NaN,82.0,2.0,0.0,CS-12 E,1984-11-15 13:44:57,...,True,", II,III,AVF",NaN,NaN,NaN,NaN,NaN,3,records100/00000/00004_lr,records500/00000/00004_hr
4,5,17448.0,19.0,1,NaN,70.0,2.0,0.0,CS-12 E,1984-11-17 10:43:15,...,True,", III,AVR,AVF",NaN,NaN,NaN,NaN,NaN,4,records100/00000/00005_lr,records500/00000/00005_hr
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
21793,21832,7954.0,63.0,0,NaN,NaN,1.0,2.0,AT-60 3,2001-05-30 14:14:25,...,True,NaN,NaN,NaN,NaN,NaN,NaN,7,records100/21000/21832_lr,records500/21000/21832_hr
21794,21833,17180.0,67.0,1,NaN,NaN,1.0,2.0,AT-60 3,2001-05-31 09:14:35,...,True,NaN,", alles,",NaN,NaN,1ES,NaN,7,records100/21000/21833_lr,records500/21000/21833_hr
21796,21835,19311.0,59.0,1,NaN,NaN,1.0,2.0,AT-60 3,2001-06-08 10:30:27,...,True,NaN,", I-AVR,",NaN,NaN,NaN,NaN,2,records100/21000/21835_lr,records500/21000/21835_hr
21797,21836,8873.0,64.0,1,NaN,NaN,1.0,2.0,AT-60 3,2001-06-09 18:21:49,...,True,NaN,NaN,NaN,NaN,SVES,NaN,8,records100/21000/21836_lr,records500/21000/21836_hr


In [10]:
enfermedad_grupo=dict(zip(df_scp.iloc[:, 0], df_scp['diagnostic_class']))

In [11]:
fs=500
pre=int(0.2*fs)#100 muestras antes del pico R
post=int(0.4*fs)#200 muestras después del pico R
MAX_LATIDOS = 10#Lo reducimos a 10 porque así hay más datos
LONGITUD_LATIDO = 300
wavelet='db6'
nivel=8
ENFERMEDADES=list(df_scp['diagnostic_class'].unique())
ENFERMEDADES=ENFERMEDADES[:-1] 

In [12]:
carpeta=Path("C:\\Users\\Ana\\Documents\\4º GITT+BA (2025-2026)\\TFG\\TFG\\Datos\\data")
#Datos 00001-00999:Faltan el 137,139,140,141,142,143,145,456,458,459,461,462
#Datos 01000-01999:No faltan datos 
#Datos 02000-02999:Faltan el 2506,2511
#Datos 03000-03999:Faltan el 3795,3798,3800,3801,3832
#Datos 04000-04999:No faltan datos
#Datos 05000-05999:Falta el 5817
#Datos 06000-06999:No faltan datos
#Datos 07000-07999:Falta el 07777, 07779,07782
#Datos 08000-08999:No faltan datos
#Datos 09000-09999:Falta el 9821,9825,9888
#Datos 10000-10999:No faltan datos
#Datos 11000-11999:Falta el 11810,11814,11815,11817,11838
#Datos 12000-12999:No faltan datos
#Datos 13000-13999:Faltan el 13791.13793,13796,13797,13799
#Datos 14000-14999:No faltan datos
#Datos 15000-15999:Falta el 15742
#Datos 16000-16999:No faltan datos
#Datos 17000-17999:No faltan datos (Los he vuelto a descargar porque daban error 56 de estos archivos)
#Datos 18000-18999:Falta el 18150
#Datos 19000-19999:No faltan datos
#Datos 20000-20999:No faltan datos
#Datos 21000-21999:Faltan el 21838-21999

In [ ]:
picos_R_pacientes = []
latidos_pacientes = []
latidos_12_pacientes = []
pacientes_validos = []

for archivo in carpeta.glob("*.hea"):
    record       = archivo.stem
    BASE         = strip_wfdb_suffix(archivo)
    p_signal, _  = wfdb.rdsamp(str(BASE))
    p_signal_t   = p_signal.T
    derivacion_I = p_signal_t[0, :]

    # Detectamos picos R siempre sobre la derivación I
    Rloc    = detectar_picos_R(derivacion_I, wavelet, nivel)
    latidos = extraer_latidos(Rloc, derivacion_I, pre, post)

    if len(latidos) < MAX_LATIDOS:
        continue

    latidos         = latidos[:MAX_LATIDOS]
    matriz_paciente = construir_matriz_paciente(latidos, MAX_LATIDOS, LONGITUD_LATIDO)

    # Extraemos latidos de las 12 derivaciones con los mismos picos R
    latidos_12 = []
    for d in range(12):
        derivacion_d = p_signal_t[d, :]
        latidos_d    = extraer_latidos(Rloc, derivacion_d, pre, post)
        if len(latidos_d) < MAX_LATIDOS:
            matriz_d = np.zeros((MAX_LATIDOS, LONGITUD_LATIDO))
        else:
            latidos_d = latidos_d[:MAX_LATIDOS]
            matriz_d  = construir_matriz_paciente(latidos_d, MAX_LATIDOS, LONGITUD_LATIDO)
        latidos_12.append(matriz_d)
    latidos_12 = np.array(latidos_12)  # (12, MAX_LATIDOS, LONGITUD_LATIDO)

    picos_R_pacientes.append(Rloc)
    latidos_pacientes.append(matriz_paciente)
    latidos_12_pacientes.append(latidos_12)
    pacientes_validos.append(record)

print(f"Pacientes válidos: {len(pacientes_validos)}")

Pacientes válidos: 17358


In [16]:
print(latidos_12.shape)

(12, 10, 300)


In [ ]:
X = []
Y = []
pacientes_validos_final = []
latidos_pacientes_final = []
latidos_12_pacientes_final = []
picos_R_pacientes_final = []

for i, record in enumerate(pacientes_validos):
    fila = df_pacientes.loc[
        df_pacientes['filename_hr'].str.contains(record, case=False, na=False)
    ]
    if len(fila) == 0:
        continue

    latidos = latidos_pacientes[i]
    latidos_12 = latidos_12_pacientes[i]
    edad = fila['age'].values[0]
    sexo = 1.0 if fila['sex'].values[0] == 1 else 0.0

    # Megavector solo derivación I + edad y sexo
    megavector = latidos.flatten()
    X.append(np.append(megavector, [edad, sexo]))

    etiqueta = fila['scp_codes'].values[0]
    y = vector_etiqueta_grupo(etiqueta, enfermedad_grupo, ENFERMEDADES)
    Y.append(y)

    pacientes_validos_final.append(record)
    latidos_pacientes_final.append(latidos_pacientes[i])
    latidos_12_pacientes_final.append(latidos_12_pacientes[i])
    picos_R_pacientes_final.append(picos_R_pacientes[i])

X = np.array(X)
Y = np.array(Y)

print(f"Pacientes finales: {len(pacientes_validos_final)}")
print("Shape X:", X.shape)     #Este solo tiene informacion de la derivación I
print("Shape Y:", Y.shape)

np.save('X_raw.npy',X)
np.save('Y.npy',Y)
np.save('picos_R.npy',np.array(picos_R_pacientes_final,dtype=object), allow_pickle=True)
np.save('latidos.npy',np.array(latidos_pacientes_final,dtype=object), allow_pickle=True)
np.save('latidos_12.npy',np.array(latidos_12_pacientes_final, dtype=object),allow_pickle=True)
np.save('pacientes_validos.npy', np.array(pacientes_validos_final),allow_pickle=True)
print("Guardado correctamente")

Pacientes finales: 16984
Shape X:     (16984, 3002)
Shape Y:     (16984, 5)


MemoryError: 